# Projet Machine Learning — Prédiction du Chômage des Diplômés Tunisiens
## Phase 3 : Prétraitement des Données et Feature Engineering

**Auteur :** Boudriga Ahmed  
**Dataset :** Enquête Nationale Population et Emploi 2017 (ENPE 2017) — INS Tunisie  
**Source :** https://www.ins.tn/enquetes/enquete-nationale-sur-la-population-et-lemploi-2017  
**Date :** 2025–2026

---

### Objectif de cette phase

Pipeline complet de prétraitement :
1. **Filtrage** — codes invalides et âge < 15
2. **Variable cible** — 3 classes : Employé / Chômeur / Inactif
3. **Sélection des features** — 9 variables ENPE
4. **Feature Engineering** — 5 nouvelles variables
5. **Imputation** — médiane
6. **Normalisation** — StandardScaler
7. **Split** — 80/20 stratifié
8. **Équilibrage** — class_weight (compatible toutes versions sklearn)

> **Codes INS confirmés :** `V_0_244_i` : 1=Chômeur, 2=Employé, 3=Inactif | `V_1_204_i` : 1=Masculin, 2=Féminin


In [14]:
# ── CELLULE 0 : Connexion Google Drive ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Définir le dossier projet sur Drive
PROJET_DIR = "/content/drive/MyDrive/MachineLearningProject"

# Créer les sous-dossiers si nécessaire
os.makedirs(f"{PROJET_DIR}/notebooks", exist_ok=True)
os.makedirs(f"{PROJET_DIR}/data",      exist_ok=True)
os.makedirs(f"{PROJET_DIR}/models",    exist_ok=True)

# Se placer dans le dossier projet
os.chdir(PROJET_DIR)

print(f"✅ Drive monté")
print(f"✅ Dossier actuel : {os.getcwd()}")
print(f"✅ Fichiers disponibles :")
for f in sorted(os.listdir(PROJET_DIR)):
    print(f"   → {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive monté
✅ Dossier actuel : /content/drive/MyDrive/MachineLearningProject
✅ Fichiers disponibles :
   → Amelioration_Tentative
   → Data
   → ENPE_2017.csv
   → README.md
   → data
   → graph
   → models
   → notebooks
   → requirements.txt
   → src


---
## 1. Importation des Librairies


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight

print("✅ Imports OK")


✅ Imports OK


---
## 2. Chargement du Dataset

Rechargement du CSV sauvegardé en Phase 1.


In [3]:
df = pd.read_csv("ENPE_2017.csv")
print(f"✅ Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")


✅ Dataset chargé : 452,928 lignes × 20 colonnes


---
## 3. Étape 1 — Filtrage

| Code V_0_244_i | Signification | Action |
|---|---|---|
| 1 | Chômeur | ✅ Conservé |
| 2 | Employé | ✅ Conservé |
| 3 | Inactif | ✅ Conservé |
| 4 | Moins de 15 ans | ❌ Exclu |
| 9 | Non déclaré | ❌ Exclu |


In [4]:
print("\n── ÉTAPE 1 : Filtrage ──────────────────────────────────────")
df = df[df["V_0_244_i"].isin([1.0, 2.0, 3.0])].copy()
print(f"Après filtrage codes invalides (4,9) : {len(df):,} lignes")
df = df[df["V_210tr"] >= 15].copy()
print(f"Après filtrage âge < 15 ans          : {len(df):,} lignes")
print(f"\nDistribution V_0_244_i après filtrage :")
print(df["V_0_244_i"].value_counts().sort_index())



── ÉTAPE 1 : Filtrage ──────────────────────────────────────
Après filtrage codes invalides (4,9) : 327,480 lignes
Après filtrage âge < 15 ans          : 295,411 lignes

Distribution V_0_244_i après filtrage :
V_0_244_i
1.0     69220
2.0     96198
3.0    129993
Name: count, dtype: int64


---
## 4. Étape 2 — Construction de la Variable Cible (3 Classes)

| Code INS | Statut | Encodage target |
|---|---|---|
| 2 | Employé | 0 |
| 1 | Chômeur | 1 |
| 3 | Inactif | 2 |

> **Codes confirmés par diagnostic :** V_0_244_i=1 → Chômeurs (116 925), V_0_244_i=2 → Employés (69 522)


In [5]:
print("\n── ÉTAPE 2 : Construction variable cible 3 CLASSES ────────")

# Codes RÉELS confirmés par diagnostic :
# V_0_244_i = 1 → Chômeur  |  V_0_244_i = 2 → Employé  |  V_0_244_i = 3 → Inactif
mapping_cible = {1.0: "Chômeur", 2.0: "Employé", 3.0: "Inactif"}
df["statut"] = df["V_0_244_i"].map(mapping_cible)

mapping_encode = {"Employé": 0, "Chômeur": 1, "Inactif": 2}
df["target"] = df["statut"].map(mapping_encode)

print("Variable cible construite :")
for statut, code in mapping_encode.items():
    count = (df["target"] == code).sum()
    pct   = count / len(df) * 100
    barre = "█" * int(pct / 2)
    print(f"  {code} = {statut:<10} : {count:>8,}  ({pct:5.1f}%)  {barre}")



── ÉTAPE 2 : Construction variable cible 3 CLASSES ────────
Variable cible construite :
  0 = Employé    :   96,198  ( 32.6%)  ████████████████
  1 = Chômeur    :   69,220  ( 23.4%)  ███████████
  2 = Inactif    :  129,993  ( 44.0%)  ██████████████████████


---
## 5. Étape 3 — Sélection des Features

| Variable | Description | Justification |
|---|---|---|
| V_9_10_i | Gouvernorat | Disparités régionales (EDA 2) |
| V_9_11_1 | Milieu urbain/rural | Impact emploi (EDA 7) |
| V_1_203_i | Lien de parenté | Position dans le ménage |
| V_1_204_i | Sexe | Asymétrie de genre (EDA 3) |
| V_1_205_i | Situation matrimoniale | Corrélée au statut |
| V_210tr | Âge | Facteur clé (EDA 5) |
| V_1_225_i | Niveau d'instruction | Paradoxe du diplôme (EDA 4) |
| V_4_321_i | Secteur d'activité | Pour actifs occupés |
| V_4_325_i | CSP | Pour actifs occupés |


In [6]:
print("\n── ÉTAPE 3 : Sélection des features ────────────────────────")

FEATURES = [
    "V_9_10_i",  "V_9_11_1",  "V_1_203_i", "V_1_204_i", "V_1_205_i",
    "V_210tr",   "V_1_225_i", "V_4_321_i", "V_4_325_i"
]
FEATURES = [f for f in FEATURES if f in df.columns]
print(f"Features disponibles ({len(FEATURES)}) : {FEATURES}")

print("\nValeurs manquantes par feature :")
for f in FEATURES:
    n_nan = df[f].isnull().sum()
    pct   = n_nan / len(df) * 100
    print(f"  {f:<15} : {n_nan:>7,} NaN  ({pct:.1f}%)")



── ÉTAPE 3 : Sélection des features ────────────────────────
Features disponibles (9) : ['V_9_10_i', 'V_9_11_1', 'V_1_203_i', 'V_1_204_i', 'V_1_205_i', 'V_210tr', 'V_1_225_i', 'V_4_321_i', 'V_4_325_i']

Valeurs manquantes par feature :
  V_9_10_i        :       0 NaN  (0.0%)
  V_9_11_1        :       0 NaN  (0.0%)
  V_1_203_i       :       0 NaN  (0.0%)
  V_1_204_i       :       0 NaN  (0.0%)
  V_1_205_i       :       0 NaN  (0.0%)
  V_210tr         :       0 NaN  (0.0%)
  V_1_225_i       :       0 NaN  (0.0%)
  V_4_321_i       : 192,517 NaN  (65.2%)
  V_4_325_i       : 192,517 NaN  (65.2%)


---
## 6. Étape 4 — Feature Engineering

| Feature créée | Basée sur | Justification |
|---|---|---|
| `region_interieure` | V_9_10_i | Chômage systématiquement plus élevé (EDA 6) |
| `groupe_age` | V_210tr | 4 tranches d'âge |
| `diplome_superieur` | V_1_225_i | Paradoxe du diplôme (EDA 4) |
| `femme_region_int` | V_1_204_i × region | Groupe doublement vulnérable |
| `jeune_diplome` | V_210tr × V_1_225_i | Profil le plus exposé |


In [7]:
print("\n── ÉTAPE 4 : Feature Engineering ───────────────────────────")

REGIONS_INTERIEURES = [21, 22, 23, 24, 41, 42, 43, 53, 61, 62, 63]
df["region_interieure"] = df["V_9_10_i"].isin(REGIONS_INTERIEURES).astype(int)
print(f"✅ region_interieure : {df['region_interieure'].value_counts().to_dict()}")

def groupe_age(age):
    if age < 25:   return 0
    elif age < 35: return 1
    elif age < 50: return 2
    else:          return 3

df["groupe_age"]        = df["V_210tr"].apply(groupe_age)
df["diplome_superieur"] = (df["V_1_225_i"] == 4.0).astype(int)
df["femme_region_int"]  = ((df["V_1_204_i"] == 2.0) & (df["region_interieure"] == 1)).astype(int)
df["jeune_diplome"]     = ((df["V_210tr"] < 35) & (df["diplome_superieur"] == 1)).astype(int)

NOUVELLES_FEATURES = ["region_interieure","groupe_age","diplome_superieur","femme_region_int","jeune_diplome"]
FEATURES_FINALES   = FEATURES + NOUVELLES_FEATURES
print(f"\nTotal features finales : {len(FEATURES_FINALES)}")
print(FEATURES_FINALES)



── ÉTAPE 4 : Feature Engineering ───────────────────────────
✅ region_interieure : {0: 163905, 1: 131506}

Total features finales : 14
['V_9_10_i', 'V_9_11_1', 'V_1_203_i', 'V_1_204_i', 'V_1_205_i', 'V_210tr', 'V_1_225_i', 'V_4_321_i', 'V_4_325_i', 'region_interieure', 'groupe_age', 'diplome_superieur', 'femme_region_int', 'jeune_diplome']


---
## 7. Étape 5 — Gestion des Valeurs Manquantes

Imputation par la **médiane** — robuste aux outliers.


In [8]:
print("\n── ÉTAPE 5 : Imputation médiane ────────────────────────────")

X = df[FEATURES_FINALES].copy()
y = df["target"].copy()

print(f"Avant imputation — NaN total : {X.isnull().sum().sum():,}")
imputer   = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES_FINALES)
print(f"Après imputation  — NaN total : {X_imputed.isnull().sum().sum()}")
print(f"Shape X : {X_imputed.shape}")



── ÉTAPE 5 : Imputation médiane ────────────────────────────
Avant imputation — NaN total : 385,034
Après imputation  — NaN total : 0
Shape X : (295411, 14)


---
## 8. Étape 6 — Normalisation (StandardScaler)

Normalisation des variables continues : **moyenne=0, écart-type=1**.


In [9]:
print("\n── ÉTAPE 6 : Normalisation ─────────────────────────────────")

COLS_SCALE = ["V_210tr", "V_9_10_i"]
scaler     = StandardScaler()
X_scaled   = X_imputed.copy()
X_scaled[COLS_SCALE] = scaler.fit_transform(X_imputed[COLS_SCALE])

print(f"Colonnes normalisées : {COLS_SCALE}")
print(f"V_210tr — avant : mean={X_imputed['V_210tr'].mean():.1f}  std={X_imputed['V_210tr'].std():.1f}")
print(f"V_210tr — après : mean={X_scaled['V_210tr'].mean():.2f}  std={X_scaled['V_210tr'].std():.2f}")



── ÉTAPE 6 : Normalisation ─────────────────────────────────
Colonnes normalisées : ['V_210tr', 'V_9_10_i']
V_210tr — avant : mean=42.9  std=17.6
V_210tr — après : mean=0.00  std=1.00


---
## 9. Étape 7 — Split Train/Test (80/20 Stratifié)


In [10]:
print("\n── ÉTAPE 7 : Split 80/20 stratifié ────────────────────────")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"\nDistribution y_train :")
for code, nom in {0:"Employé", 1:"Chômeur", 2:"Inactif"}.items():
    count = (y_train == code).sum()
    pct   = count / len(y_train) * 100
    print(f"  {nom:<10} : {count:>8,}  ({pct:.1f}%)")



── ÉTAPE 7 : Split 80/20 stratifié ────────────────────────
X_train : (236328, 14)  |  X_test : (59083, 14)

Distribution y_train :
  Employé    :   76,958  (32.6%)
  Chômeur    :   55,376  (23.4%)
  Inactif    :  103,994  (44.0%)


---
## 10. Étape 8 — Équilibrage des Classes (class_weight)

Au lieu de SMOTE (incompatible avec NumPy 2.x), nous utilisons `class_weight='balanced'`.
Cette approche est **académiquement valide** et évite la création de données synthétiques.


In [11]:
print("\n── ÉTAPE 8 : Équilibrage (class_weight) ────────────────────")

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))

print("Poids des classes calculés :")
for code, weight in class_weight_dict.items():
    nom   = {0:"Employé", 1:"Chômeur", 2:"Inactif"}[code]
    count = (y_train == code).sum()
    print(f"  {nom:<10} : {count:>8,} individus  → poids = {weight:.3f}")

# Compatibilité avec Phase 4
X_train_res = X_train.values if hasattr(X_train, 'values') else X_train
y_train_res = y_train.values if hasattr(y_train, 'values') else y_train
X_test_np   = X_test.values  if hasattr(X_test,  'values') else X_test

print(f"\nX_train_res : {X_train_res.shape}")
print("\n✅ Équilibrage par class_weight — prêt pour Phase 4")



── ÉTAPE 8 : Équilibrage (class_weight) ────────────────────
Poids des classes calculés :
  Employé    :   76,958 individus  → poids = 1.024
  Chômeur    :   55,376 individus  → poids = 1.423
  Inactif    :  103,994 individus  → poids = 0.758

X_train_res : (236328, 14)

✅ Équilibrage par class_weight — prêt pour Phase 4


---
## 11. Sauvegarde des Données Prétraitées


In [12]:
import joblib, os

# ── Chemin vers le dossier models ──
MODELS_DIR = f"{PROJET_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Sauvegarder dans models/
np.save(f"{MODELS_DIR}/X_train_res.npy", X_train_res)
np.save(f"{MODELS_DIR}/X_test.npy",      X_test_np)
np.save(f"{MODELS_DIR}/y_train_res.npy", y_train_res)
np.save(f"{MODELS_DIR}/y_test.npy",      y_test.values if hasattr(y_test, 'values') else y_test)

joblib.dump(scaler,           f"{MODELS_DIR}/scaler.pkl")
joblib.dump(imputer,          f"{MODELS_DIR}/imputer.pkl")
joblib.dump(FEATURES_FINALES, f"{MODELS_DIR}/features.pkl")

print("✅ Fichiers sauvegardés dans /models/ :")
for f in sorted(os.listdir(MODELS_DIR)):
    print(f"   → {f}")

✅ Fichiers sauvegardés dans /models/ :
   → X_test.npy
   → X_train_res.npy
   → features.pkl
   → imputer.pkl
   → modele_final.json
   → modele_final.pkl
   → resultats_modeles.pkl
   → scaler.pkl
   → y_test.npy
   → y_train_res.npy


---
## 12. Résumé de la Phase 3


In [13]:
print("\n" + "=" * 55)
print("  RÉSUMÉ PHASE 3 — PREPROCESSING 3 CLASSES")
print("=" * 55)
print(f"  Dataset final          : {len(df):,} individus")
print(f"  Problème               : Classification 3 classes")
print(f"  Cible                  : Employé / Chômeur / Inactif")
print(f"  Features totales       : {len(FEATURES_FINALES)}")
print(f"    → Originales         : {len(FEATURES)}")
print(f"    → Créées (FE)        : {len(NOUVELLES_FEATURES)}")
print(f"  X_train                : {X_train_res.shape}")
print(f"  X_test                 : {X_test_np.shape}")
print(f"\n  Pipeline :")
print(f"    1. Filtrage codes invalides + âge < 15")
print(f"    2. Variable cible 3 classes (codes INS corrigés)")
print(f"    3. 9 features originales sélectionnées")
print(f"    4. 5 nouvelles features créées")
print(f"    5. Imputation médiane")
print(f"    6. StandardScaler")
print(f"    7. Split 80/20 stratifié")
print(f"    8. class_weight balanced (compatible NumPy 2.x)")
print("=" * 55)
print("\n✅ Phase 3 terminée → Phase 4 (Modélisation) !")



  RÉSUMÉ PHASE 3 — PREPROCESSING 3 CLASSES
  Dataset final          : 295,411 individus
  Problème               : Classification 3 classes
  Cible                  : Employé / Chômeur / Inactif
  Features totales       : 14
    → Originales         : 9
    → Créées (FE)        : 5
  X_train                : (236328, 14)
  X_test                 : (59083, 14)

  Pipeline :
    1. Filtrage codes invalides + âge < 15
    2. Variable cible 3 classes (codes INS corrigés)
    3. 9 features originales sélectionnées
    4. 5 nouvelles features créées
    5. Imputation médiane
    6. StandardScaler
    7. Split 80/20 stratifié
    8. class_weight balanced (compatible NumPy 2.x)

✅ Phase 3 terminée → Phase 4 (Modélisation) !
